---
title: "4.2 Correlation and simple OLS"
---

<div class="lesson-tools">
<a href="https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/04-regression/02-correlation-ols.ipynb" class="btn btn-primary">Open in Colab</a>
<a href="https://ningzhanglab.github.io/Python-for-Beginners/downloads/04-regression/02-correlation-ols.ipynb" class="btn btn-outline-secondary">Download notebook</a>
</div>

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [4.1 Descriptive statistics and a Table 1](../04-regression/01-descriptive-statistics.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Distinguish correlation from the slope of an OLS regression.
- Fit a model with a formula and interpret a log-income coefficient in years.
- Read uncertainty and model fit without claiming causation.

## Why it matters

**Correlation** describes the strength and direction of a linear association, on a scale from −1 to +1. **Ordinary least squares (OLS)** fits coefficients by minimizing the sum of squared vertical prediction errors, called residuals. A coefficient has units; correlation has none. Neither statistic by itself identifies cause and effect. We first fit made-up country measurements, then estimate the running question on one real historical cross-section.

## Worked example

### Prepare the analysis libraries

Run this prepared cell **before importing other analysis libraries**. It uses the versions generated from the course's single dependency file. If needed, it installs free packages inside this temporary Python runtime, not on your own computer. An internet connection is needed for installation. If it reports that a restart is required, restart the Colab session and run from the top; mixing already loaded old code with newly installed libraries can produce misleading errors.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared analysis-library setup (run first)"
# Generated from requirements.txt; installs only in this Python runtime.
import importlib.metadata
from pathlib import Path
import subprocess
import sys

requested = ['statsmodels==0.15.0']
needs_install = False
for requirement in requested:
    name, version = requirement.split("==")
    try:
        needs_install = needs_install or importlib.metadata.version(name) != version
    except importlib.metadata.PackageNotFoundError:
        needs_install = True
if needs_install:
    loaded = any(name in sys.modules for name in ["numpy", "pandas", "scipy", "statsmodels", "linearmodels"])
    constraints = Path("scratch/course-analysis-constraints.txt")
    constraints.parent.mkdir(parents=True, exist_ok=True)
    constraints.write_text('# Python 3.12; tested on macOS. CI also installs this file on Ubuntu.\n# Direct dependencies and their resolved transitive dependencies are pinned.\n# Add a library only when an available lesson or a course check needs it.\nappnope==1.0.0; sys_platform == "darwin"\nasttokens==3.0.2\nattrs==26.1.0\ncomm==0.2.3\ncontourpy==1.4.0\ncycler==0.12.1\ndebugpy==1.8.22\net_xmlfile==2.0.0\nexecuting==2.3.0\nfastjsonschema==2.22.3\nfonttools==4.68.0\nipykernel==7.4.0\nipython==9.18.1\nipython_pygments_lexers==1.1.1\njedi==0.20.1\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\njupyter_client==8.10.0\njupyter_core==5.9.1\nkiwisolver==1.5.1\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nnbclient==0.11.0\nnbformat==5.11.1\nnest-asyncio2==1.7.4\nnumpy==2.5.4\nopenpyxl==3.1.5\npackaging==26.3\npandas==3.1.1\nparso==0.8.7\npexpect==4.9.0; sys_platform != "win32"\npillow==13.0.0\nplatformdirs==4.14.5\nprompt_toolkit==3.0.53\npsutil==7.2.2\nptyprocess==0.7.0; sys_platform != "win32"\npure_eval==0.2.4\nPygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\nPyYAML==6.0.3\npyzmq==27.3.0\nreferencing==0.37.0\nrpds-py==2026.9.1\nstack-data==0.6.3\ntornado==6.6.1\ntraitlets==5.16.1\ntyping_extensions==4.16.0\ntzdata==2026.6\nwcwidth==0.9.4\n\n# Statistics and fixed-effects models, with their newly required dependencies.\nformulaic==1.2.2\ninterface_meta==2.0.1\nlinearmodels==7.1\nmypy_extensions==1.2.0\nnarwhals==2.28.0\npatsy==1.0.3\npyhdfe==0.2.1\nscipy==1.18.1\nstatsmodels==0.15.0\nwrapt==2.5.2\n', encoding='utf-8')
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--constraint", str(constraints), *requested], check=True)
    if loaded:
        raise RuntimeError("Packages installed. Restart the notebook session, then run all cells from the top so updated libraries are used together.")
print("Analysis libraries ready:", ", ".join(requested))

Look for **Analysis libraries ready**. In the pinned course environment this cell installs nothing. Colab may take a few minutes the first time. The current course environment uses Python 3.12; an older runtime that cannot install the pinned dependencies needs a current Python runtime rather than arbitrary version substitutions.

### Fit a small formula model

Import the formula interface as `smf`. In a formula, the variable left of `~` is the **outcome** being described; terms on the right are **predictors**. `smf.ols` defines the model and `.fit()` estimates its coefficients. An intercept (a constant term) is included by default. This small example uses made-up numbers, and its tiny sample is for commands rather than inference.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
small = pd.DataFrame({
    "country": ["Northland", "Northland", "Eastria", "Eastria", "Southport", "Southport"],
    "year": [2000, 2020, 2000, 2020, 2000, 2020],
    "gdp_pc": [8000, 15000, 21000, 30000, 2500, 4200],
    "life_exp": [68.0, 73.5, 76.0, 80.1, 58.2, 64.0],
})
small["gdp_thousands"] = small["gdp_pc"] / 1000
toy_model = smf.ols("life_exp ~ gdp_thousands", data=small).fit()
print(toy_model.params.round(3))

The slope describes the fitted change in life-expectancy years for 1000 more made-up dollars of GDP per person. The intercept is the fitted outcome at zero of this predictor; it need not describe a meaningful observed country. Correlation would instead be a unitless summary.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Keep the sample and transformations explicit

Select 2007 and the required columns, count missing values, then keep complete observations for the formula. Verify positive GDP before creating its natural logarithm. `.corr()` gives the correlation for the two numeric columns.

In [ ]:
countries = pd.read_csv(data_file)
sample = countries.loc[countries["year"] == 2007, ["country", "life_exp", "gdp_pc"]].copy()
print(sample.isna().sum())
sample = sample.dropna(subset=["life_exp", "gdp_pc"])
print("Positive GDP:", sample["gdp_pc"].gt(0).all())
sample["log_gdp_pc"] = np.log(sample["gdp_pc"])
correlation = sample["life_exp"].corr(sample["log_gdp_pc"])
print("Analysis observations:", len(sample))
print("Correlation:", round(correlation, 3))

All 142 observations remain. The correlation is about 0.809. It describes a strong positive linear association with **log** income in this sample; it is not a correlation of individual people's incomes and lifespans. Missingness rules should be decided and reported rather than left to silent row dropping inside a model.

### Estimate a descriptive regression

Fit life expectancy on log GDP per person. `missing="raise"` refuses unexpected missing formula inputs after your explicit sample preparation. `cov_type="HC1"` asks for heteroskedasticity-robust uncertainty, explained in lesson 4.4; `use_t=True` requests t-based approximate inference. These choices do not change the OLS coefficient estimates.

In [ ]:
model = smf.ols("life_exp ~ log_gdp_pc", data=sample, missing="raise").fit(cov_type="HC1", use_t=True)
results = pd.DataFrame({"estimate": model.params, "robust_se": model.bse})
print(results.round(3))
print("N:", int(model.nobs))
print("R-squared:", round(model.rsquared, 3))

The log-income slope is about 7.203 **years per natural-log unit**. The intercept is about 4.950, the fitted value at GDP per person of one dollar (log 1 = 0), far outside this sample's observed range. Do not interpret that intercept as life expectancy at zero income. **R-squared** is the fraction of sample outcome variation described by this fitted model with an intercept; a high value is not proof of causation or a correct model.

### Translate the slope into a readable comparison

A one-percent income increase changes log income by `log(1.01)`. A doubling changes it by `log(2)`. Multiply those changes by the fitted slope to express the model's comparison in years. This is a level-log model: the outcome itself is not logged.

In [ ]:
slope = model.params["log_gdp_pc"]
one_percent_comparison = slope * np.log(1.01)
doubling_comparison = slope * np.log(2)
print("1% higher GDP per person, fitted difference in years:", round(one_percent_comparison, 3))
print("Twice the GDP per person, fitted difference in years:", round(doubling_comparison, 3))

The fitted differences are about 0.072 years for 1% and 4.993 years for a doubling. They are descriptive model comparisons, not effects of giving a country more income. Omitted differences, reverse causation and measurement issues can influence the association. Extrapolating beyond observed GDP levels adds another unsupported assumption.

### Read the interval as an uncertainty procedure

A **standard error** estimates the uncertainty of a coefficient under a sampling/error model. It is not the spread of individual observations. `conf_int()` returns confidence-interval endpoints using the chosen uncertainty settings. Keep the full-precision values for calculations and round only the display.

In [ ]:
interval = model.conf_int().loc["log_gdp_pc"]
print("Approximate 95% interval for the slope:", interval.round(3).tolist())
print("Mean residual:", round(float(model.resid.mean()), 10))

The approximate interval is about 6.543 to 7.863 years per log unit. Under its assumptions, the interval procedure aims for 95% coverage across repeated samples; it does not assign a 95% probability to a fixed parameter after seeing this interval. This historical teaching extract is not a designed probability sample of all countries or people, so these are model-based uncertainty calculations, not certified global-population statements. A near-zero average residual is an OLS property with an intercept, not proof that all predictions are accurate.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Compare raw and log income correlation

Compute life expectancy's correlation with GDP per person in dollars. Compare it with the stored log-income correlation.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
raw_correlation = sample["life_exp"].corr(sample["gdp_pc"])
print("Dollars:", round(raw_correlation, 3))
print("Log income:", round(correlation, 3))

The values differ because correlation describes a linear association in the variables actually supplied. A transformation changes that question; select it for substantive and diagnostic reasons rather than to maximize a correlation.

</details>

### 2. Translate a 10% difference

Use the fitted log-income slope to express 10% higher GDP per person in years of fitted life expectancy. Store the comparison as `ten_percent_comparison`.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
ten_percent_comparison = slope * np.log(1.10)
print(round(ten_percent_comparison, 3))

The result is about 0.687 years. This is a fitted difference for the same descriptive model, not a causal or individual-level effect.

</details>

### 3. Get a fitted level in range

Use `.predict` with a one-row DataFrame to obtain fitted life expectancy at GDP per person of 10000 dollars. Supply the predictor on the formula's log scale.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
prediction = model.predict(pd.DataFrame({"log_gdp_pc": [np.log(10000)]}))
print(round(float(prediction.iloc[0]), 2))

The fitted level is about 71.29 years. This is a conditional fitted mean, not a prediction interval for an individual country and not a guarantee about a person.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### A formula term is absent

A formula library wraps an undefined predictor in its own informative error. PatsyError is the formula parser's error type.

In [ ]:
try:
    from patsy import PatsyError
    smf.ols("life_exp ~ GDP_typo", data=sample).fit()
except PatsyError as error:
    print(f"{type(error).__name__}: {error}")

Read the formula term and indicated position. Here the intended predictor is log_gdp_pc, not a differently named raw dollar variable.

### Prediction inputs do not match the formula

A formula fitted with log_gdp_pc needs that named transformed predictor.

In [ ]:
try:
    model.predict(pd.DataFrame({"gdp_pc": [10000]}))
except PatsyError as error:
    print(f"{type(error).__name__}: {error}")

Construct the predictor with its correct name and transformation. Supplying raw dollars in a column labelled log_gdp_pc would run, but would give a meaningless prediction; no error message can substitute for checking units.

## Recap

- Correlation is unitless; regression coefficients depend on outcome and predictor units.
- Formulas include an intercept unless you specify otherwise.
- For a level-log model, multiply the slope by the log of an income ratio.
- Coefficient uncertainty, fit and causal identification are separate questions.

**Next:** [4.3 Multiple regression, categories and interactions](../04-regression/03-multiple-regression.ipynb).

> **Coming from Stata / R / Excel?**
>
> The formula resembles R's lm syntax. Stata users can think of life_exp as the dependent variable and log_gdp_pc as a regressor. The intercept is automatic here; the covariance setting must still be chosen explicitly.